# Kaggle 1/2 — Extraer rostros de FaceForensics++ (C23)

**Configuración (panel derecho → *Session options*):**
- **Accelerator:** None (solo CPU; la GPU no ayuda aquí y gasta cuota).
- **Internet:** On (para clonar el repositorio; requiere cuenta verificada con el celular).
- **Input:** *Add Input* → buscar `ff-c23` (dataset de xdxd003) → *Add*.

**Cómo correrlo:** *Save Version* → **Save & Run All (Commit)**. Corre en segundo plano aunque cierres el navegador. Al terminar, la pestaña *Output* de la versión tiene:
- `faces_ffpp.zip`: los rostros extraídos.
- `splits/ffpp/`: train / val / test sin fugas de identidad.

Esa salida es la entrada del notebook 2.

In [ ]:
import os, subprocess
REPO = 'https://github.com/Julianchu850/proyecto-redes-neuronales-deteccion-deepfake.git'
if not os.path.exists('/tmp/repo'):
    subprocess.run(['git', 'clone', '-q', REPO, '/tmp/repo'], check=True)
else:
    subprocess.run(['git', '-C', '/tmp/repo', 'pull', '-q'], check=True)
%cd /tmp/repo
!git log --oneline -1

In [ ]:
import cv2, glob, os
print('OpenCV', cv2.__version__, '| CPUs:', os.cpu_count())
assert hasattr(cv2, 'FaceDetectorYN'), 'Se necesita OpenCV >= 4.5.4'

# Ubica el dataset montado (la ruta exacta cambia según la versión de Kaggle)
candidatos = glob.glob('/kaggle/input/**/ff-c23', recursive=True) + glob.glob('/kaggle/input/*')
RAIZ = next(c for c in candidatos if glob.glob(f'{c}/**/*.mp4', recursive=True))
print('Dataset en:', RAIZ)
!find "{RAIZ}" -maxdepth 3 -type d | head -30

## Prueba rápida con 20 videos

Confirma que el script reconoce la estructura de carpetas antes de procesar todo. Revisa la línea `Videos encontrados:` y las caras de la figura.

In [ ]:
!python -m src.data.extraer_rostros --dataset ffpp --raiz "{RAIZ}" --salida /tmp/prueba --limite 20 --procesos {os.cpu_count()}

import matplotlib.pyplot as plt
from PIL import Image
pngs = sorted(glob.glob('/tmp/prueba/**/*.png', recursive=True))[::7][:8]
fig, ax = plt.subplots(1, len(pngs), figsize=(2 * len(pngs), 2.2))
for a, p in zip(ax, pngs):
    a.imshow(Image.open(p)); a.set_title(p.split('/')[-3], fontsize=7); a.axis('off')
plt.show()
!rm -rf /tmp/prueba

## Extracción completa

32 frames por video real y 8 por falso (clases balanceadas). FaceShifter, si viene, se extrae pero nunca se usa para entrenar.

In [ ]:
!python -m src.data.extraer_rostros --dataset ffpp --raiz "{RAIZ}" --salida /tmp/faces/ffpp --procesos {os.cpu_count()}

In [ ]:
!python -m src.data.crear_splits --dataset ffpp --rostros /tmp/faces/ffpp --salida /kaggle/working/splits/ffpp

## Empaquetar la salida

Un solo `.zip` sin recomprimir (`-0`): los PNG ya están comprimidos, y Kaggle maneja mucho mejor un archivo grande que decenas de miles pequeños.

In [ ]:
%cd /tmp/faces
!zip -q -r -0 /kaggle/working/faces_ffpp.zip ffpp -x '*/_listo.csv'
%cd /tmp/repo
!ls -lh /kaggle/working/faces_ffpp.zip
!du -sh /tmp/faces/ffpp

In [ ]:
import pandas as pd
m = pd.read_csv('/tmp/faces/ffpp/manifiesto.csv', dtype={'video_id': str})
print(m.groupby('metodo').agg(rostros=('ruta', 'size'), videos=('video', 'nunique')))
muestra = m.groupby('metodo').sample(4, random_state=0)
filas = len(muestra) // 4
fig, ax = plt.subplots(filas, 4, figsize=(8, 2.2 * filas))
for a, (_, f) in zip(ax.flat, muestra.iterrows()):
    a.imshow(Image.open(f'/tmp/faces/ffpp/{f.ruta}')); a.set_title(f.metodo, fontsize=8); a.axis('off')
plt.tight_layout(); plt.show()